# Agents

에이전트는 언어 모델과 도구를 결합하여 작업에 대해 추론하고, 사용할 도구를 결정하고, 반복적으로 솔루션을 향해 노력하는 시스템을 만듭니다.

ReAct("추론 + 행동") 패턴을 따르며, 간단한 추론 단계와 타겟 도구 호출을 번갈아가며 수행하고, 그 결과 관찰 결과를 후속 결정에 반영하여 최종 답변을 제공합니다.

※ 주체(Excutor), 추론(Reasoning), 행동(Act)

> https://docs.langchain.com/oss/python/langchain/agents

| 비교 항목 | LCEL (LangChain Expression Language) | Agent (에이전트) |
| :--- | :--- | :--- |
| **핵심 개념** | 개발자가 정의한 고정된 실행 경로(Chain) | LLM이 다음에 수행할 작업을 스스로 결정 |
| **제어 흐름** | **결정적 (Deterministic)**. 입력부터 출력까지의 단계가 미리 정해짐. | **비결정적 (Non-deterministic)**. 추론에 따라 단계가 동적으로 변함. |
| **의사결정 주체** | **개발자**. 어떤 도구를 어떤 순서로 쓸지 코드로 명시함. | **LLM**. 질문에 따라 어떤 도구를 쓸지 스스로 판단함. |
| **복잡도/관리** | 구조가 명확하여 디버깅과 유지보수가 용이함 | 추론 과정이 복잡하고 예상치 못한 결과가 발생할 수 있음 |
| **실행 속도** | 정해진 단계만 수행하므로 상대적으로 빠름 | 여러 번의 추론 및 도구 호출로 인해 상대적으로 느림 |
| **주요 활용** | RAG, 데이터 추출, 챗봇의 고정 시나리오 | 데이터 분석, 웹 검색 기반 문제 해결, 자율적 작업 수행 |
| **비유** | **요리 레시피**: 정해진 순서대로 조리 | **숙련된 요리사**: 재료 상황에 맞춰 유동적으로 조리 |

```bash
uv add langchain
uv add langchain-google-genai
```

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### TooL 작성법

1. 함수 작성
2. 타입 힌팅
3. docstring 작성
4. @tool 데코레이터 적용

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """특정 도시의 날씨를 가져옵니다."""
    # return f"{city}은 맑습니다." # 추가적인 자연어 답변이 필요하지 않다고 판단하여 출력을 생성하지 않을 수 있음
    return f"{city}: 맑음"

### 정적 모델(Static model) 에이전트 생성

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[get_weather],
    system_prompt="당신은 유능하고 친절한 조수입니다.",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "부산의 날씨를 알려주세요."}]}
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


{'messages': [HumanMessage(content='부산의 날씨를 알려주세요.', additional_kwargs={}, response_metadata={}, id='6e2727b7-3fbf-481f-865f-ad65126b6ba9'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "\\ubd80\\uc0b0"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba209-e86b-7282-8fe2-16a707ac4215-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '부산'}, 'id': '65b78b74-0ec9-404e-b55d-23e18b8f7cfa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 15, 'total_tokens': 81, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='부산: 맑음', name='get_weather', id='dd6eb8a8-2106-41e4-b617-8b50ca996149', tool_call_id='65b78b74-0ec9-404e-b55d-23e18b8f7cfa'),
  AIMessage(content='부산은 맑습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'g

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

# 모델 상세 설정
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    temperature=0.1,
    max_tokens=100,
    timeout=30
)

agent = create_agent(
    model,
    tools=[get_weather],
    system_prompt="당신은 유능하고 친절한 조수입니다.",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "부산의 날씨를 알려주세요."}]}
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


{'messages': [HumanMessage(content='부산의 날씨를 알려주세요.', additional_kwargs={}, response_metadata={}, id='157672c8-dbf1-4734-ac20-285a4592eb88'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'MALFORMED_FUNCTION_CALL', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba20c-2ab2-78a0-9c74-7ec51643288e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 0, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}})]}

### 사칙 연산 툴

In [4]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    두 숫자를 더해 그 합을 반환합니다.

    Args:
        a: 더할 첫 번째 숫자
        b: 더할 두 번째 숫자
    """
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """
    첫 번째 숫자에서 두 번째 숫자를 뺍니다.

    Args:
        a: 빼기 연산의 대상이 되는 수 (피감수)
        b: 뺄 숫자 (감수)
    """
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """
    두 숫자를 곱해 그 결과를 반환합니다.

    Args:
        a: 곱할 첫 번째 숫자
        b: 곱할 두 번째 숫자
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """
    첫 번째 숫자를 두 번째 숫자로 나눕니다. 
    0으로 나누려고 하면 에러 메시지를 반환합니다.

    Args:
        a: 나뉘는 수 (피제수)
        b: 나누는 수 (제수)
    """
    if b == 0:
        return "Error: 0으로 나눌 수 없습니다."
    return a / b

# 툴 리스트
tools = [add, subtract, multiply, divide]

In [11]:
print(f"더하기 테스트: {add.invoke({'a': 10, 'b': 5})}")
print(f"나누기 테스트: {divide.invoke({'a': 10, 'b': 2})}")

더하기 테스트: 15
나누기 테스트: 5.0


In [7]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    temperature=0,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [8]:
agent = create_agent(model, tools)

In [9]:
# 도구 호출 하지 않을 수도 있음
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123 + 456 * 789 / 12은 뭔가요?"}]}
)

In [10]:
result

{'messages': [HumanMessage(content='123 + 456 * 789 / 12은 뭔가요?', additional_kwargs={}, response_metadata={}, id='b784aa57-09d7-40d4-9208-4595923ccc84'),
  AIMessage(content='계산 순서에 따라 123 + (456 * 789) / 12를 계산해야 합니다.\n\n먼저 456 * 789를 계산하겠습니다.', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 789, "a": 456}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba1c2-df1d-7ab0-bd1d-c11b110ecdd5-0', tool_calls=[{'name': 'multiply', 'args': {'b': 789, 'a': 456}, 'id': 'dbc6c5b8-ad2f-44df-ba2b-bd72fc2de9b5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 356, 'output_tokens': 69, 'total_tokens': 425, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='359784', name='multiply', id='59cf2ac3-5817-46ad-827e-4c5ecaaa29fd', tool_call_id='dbc6c5b8-ad2f-44df-ba2b-bd72fc2de9b5'),
  AIMessage(content='', additional_kwargs

In [62]:
agent = create_agent(
    model, 
    tools,
    system_prompt="당신은 유능하고 친절한 수학 선생님입니다. 사칙 연산 요청 시 계산 우선 순위별로 단계를 나누고 계획을 세웁니다. 순서대로 Tool을 이용해 계산한 후 최종 답변을 생성합니다.",
)

In [63]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123 + 456 * 789 / 12은 뭔가요?"}]}
)

In [64]:
result

{'messages': [HumanMessage(content='123 + 456 * 789 / 12은 뭔가요?', additional_kwargs={}, response_metadata={}, id='516422b5-2874-48ca-82df-0d2ca950dfe6'),
  AIMessage(content='계산 순서에 따라 문제를 해결해 드릴게요.\n\n1. 곱셈: 456 * 789\n2. 나눗셈: (1단계 결과) / 12\n3. 덧셈: 123 + (2단계 결과)\n\n먼저 곱셈부터 계산해 보겠습니다.', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 456, "b": 789}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b9be8-899f-7a30-b3f5-a7cf1dbfbfc0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 456, 'b': 789}, 'id': '2be1cea9-eb8f-49c6-b75d-1e987b803b9d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 407, 'output_tokens': 93, 'total_tokens': 500, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='359784', name='multiply', id='c4a1bf87-9ab4-4cb7-84ee-59828afeb406', tool_call_id='2be1cea9-eb8f-49c6-b75d-1e987b803b9d'),